# TriaGS: 커스텀 데이터셋 실험 파이프라인

**WACV 2026** | [Paper](https://arxiv.org/abs/2512.06269) | [GitHub](https://github.com/BAEJUNHAK/triags)

---

| 단계 | 설명 |
|------|------|
| **0** | 환경 설정 |
| **1** | 데이터셋 다운로드 + 변환 (샘플링 + Train/Test 분할) |
| **2** | 학습 |
| **3** | 메쉬 추출 |
| **4** | 렌더링 + 평가 |
| **5** | 시각화 |
| **6** | 결과 저장 |

---
## 0. 환경 설정

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU 필요! Runtime > Change runtime type > T4/A100 선택")

In [ ]:
import os

WORK_DIR = "/content/triags"
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/output"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/BAEJUNHAK/triags {WORK_DIR}
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q open3d trimesh mediapy scikit-image opencv-python plyfile tqdm tensorboard gdown
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q git+https://github.com/camenduru/simple-knn

In [ ]:
try:
    from diff_gaussian_rasterization import GaussianRasterizationSettings
    print("[OK] diff-gaussian-rasterization")
except ImportError as e:
    print(f"[FAIL] {e}")

try:
    import simple_knn
    print("[OK] simple-knn")
except ImportError as e:
    print(f"[FAIL] {e}")

---
## 1. 데이터셋 다운로드 & 변환

### 이미지 수 가이드

| 기존 벤치마크 | scene당 이미지 수 |
|-------------|------------------|
| DTU | 49~64장 |
| NeRF Synthetic | ~100장 |
| Tanks & Temples | 150~400장 |

| `NUM_TRAIN` 설정 | 총 사용 | 뷰 간격 | 용도 |
|---------|------|---------|------|
| `80` | 80+20=100 | ~3.6° | 빠른 실험 |
| `160` | 160+40=200 | ~1.8° | **권장** |
| `256` | 256+64=320 | ~1.1° | 고품질 |

### Train/Test 분할

```
640장 전체 → 균일 샘플링 200장 → 인터리빙 분할 (stride=5)
Train: 160장 (stride 내 앞쪽)
Test:   40장 (stride 내 마지막 = train 뷰 사이에 위치)
```

test 뷰가 train 뷰 **사이 각도**에 위치하여 interpolation 능력을 제대로 평가합니다.

### 1-1. 다운로드 & 압축 해제

In [ ]:
import gdown

ZIP_PATH = f"{DATA_DIR}/dataset.zip"
RAW_DIR = f"{DATA_DIR}/raw"

if not os.path.exists(ZIP_PATH):
    gdown.download(
        "https://drive.google.com/uc?id=1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB",
        ZIP_PATH, quiet=False
    )
else:
    print(f"이미 존재: {ZIP_PATH}")

In [ ]:
import glob

!unzip -q -o {ZIP_PATH} -d {RAW_DIR}

def find_data_root(base_dir):
    for root, dirs, files in os.walk(base_dir):
        if any(f.startswith('calib_') and f.endswith('.ini') for f in files):
            return root
    return base_dir

RAW_DATA_DIR = find_data_root(RAW_DIR)
print(f"데이터 폴더: {RAW_DATA_DIR}")
print(f"Calibration: {len(glob.glob(f'{RAW_DATA_DIR}/calib_*.ini'))}개")
print(f"RGB 이미지:  {len(glob.glob(f'{RAW_DATA_DIR}/rgb_*.png'))}개")

In [ ]:
with open(sorted(glob.glob(f"{RAW_DATA_DIR}/calib_*.ini"))[0]) as f:
    print(f.read())

### 1-2. 샘플링 + Train/Test 분할 + COLMAP 변환

In [ ]:
#@title 데이터 설정 {display-mode: "form"}

NUM_TRAIN = 160  #@param {type:"integer"}
#@markdown ↑ Train 이미지 수 (권장: 160)

NUM_TEST = 40  #@param {type:"integer"}
#@markdown ↑ Test 이미지 수 (권장: Train의 ~25%)

NUM_INIT_POINTS = 100000  #@param {type:"integer"}
#@markdown ↑ 초기 3D 포인트 수

NUM_TOTAL = NUM_TRAIN + NUM_TEST
print(f"Train: {NUM_TRAIN}장, Test: {NUM_TEST}장, 총: {NUM_TOTAL}장")
print(f"추정 뷰 간격: ~{360/NUM_TOTAL:.1f}°")

In [ ]:
import numpy as np
from pathlib import Path
import shutil

SCENE_DIR = f"{DATA_DIR}/scene"

# ============================================================
# 유틸 함수
# ============================================================
def parse_calib(filepath):
    """calib_XXXX.ini → K, R, T, width, height"""
    config = {}
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('[') or line.startswith('#'):
                continue
            key, value = line.split('=', 1)
            config[key] = value

    k_vals = list(map(float, config['k_matrix'].split(':')[-1].split(',')))
    K = np.array(k_vals).reshape(3, 3)
    r_vals = list(map(float, config['r_matrix'].split(':')[-1].split(',')))
    R = np.array(r_vals).reshape(3, 3)
    t_vals = list(map(float, config['t_vector'].split(':')[-1].split(',')))
    T = np.array(t_vals)
    return K, R, T, int(config['width']), int(config['height'])


def rotmat2qvec(R):
    """회전 행렬 → COLMAP 쿼터니언 [w, x, y, z]"""
    Rxx, Ryx, Rzx, Rxy, Ryy, Rzy, Rxz, Ryz, Rzz = R.flat
    K = np.array([
        [Rxx - Ryy - Rzz, 0, 0, 0],
        [Ryx + Rxy, Ryy - Rxx - Rzz, 0, 0],
        [Rzx + Rxz, Rzy + Ryz, Rzz - Rxx - Ryy, 0],
        [Ryz - Rzy, Rzx - Rxz, Rxy - Ryx, Rxx + Ryy + Rzz]]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K)
    qvec = eigvecs[[3, 0, 1, 2], np.argmax(eigvals)]
    if qvec[0] < 0:
        qvec *= -1
    return qvec


# ============================================================
# 1) 파일 매칭 + 인터리빙 분할
# ============================================================
def get_index(filepath):
    return Path(filepath).stem.split('_')[-1]

calib_dict = {get_index(f): f for f in sorted(glob.glob(f"{RAW_DATA_DIR}/calib_*.ini"))}
rgb_dict = {get_index(f): f for f in sorted(glob.glob(f"{RAW_DATA_DIR}/rgb_*.png"))}
all_indices = sorted(set(calib_dict.keys()) & set(rgb_dict.keys()))
N_ALL = len(all_indices)

print(f"전체 쌍: {N_ALL}개")

# 균일 샘플링
if NUM_TOTAL < N_ALL:
    sample_idx = np.linspace(0, N_ALL - 1, NUM_TOTAL, dtype=int)
    selected_indices = [all_indices[i] for i in sample_idx]
    print(f"균일 샘플링: {N_ALL}장 → {len(selected_indices)}장")
else:
    selected_indices = all_indices
    print(f"전체 사용: {len(selected_indices)}장")

# 인터리빙 분할: stride번째마다 test
stride = max(1, NUM_TOTAL // NUM_TEST)
train_indices = []
test_indices = []

for i, idx in enumerate(selected_indices):
    if (i + 1) % stride == 0:
        test_indices.append(idx)
    else:
        train_indices.append(idx)

print(f"\nTrain: {len(train_indices)}장, Test: {len(test_indices)}장")
print(f"Train 뷰 간격: ~{360/len(train_indices):.2f}°")
print(f"Test 위치: 매 {stride}번째 = train 뷰 사이")

In [ ]:
# Train/Test 분포 시각화
import matplotlib.pyplot as plt

all_pos = []
for idx in selected_indices:
    _, R, T, _, _ = parse_calib(calib_dict[idx])
    all_pos.append(-R.T @ T)
all_pos = np.array(all_pos)
train_mask = np.array([idx in train_indices for idx in selected_indices])

fig = plt.figure(figsize=(10, 5))

ax1 = fig.add_subplot(121)
ax1.scatter([int(i) for i in train_indices], [1]*len(train_indices), c='blue', s=8, alpha=0.6, label=f'Train ({len(train_indices)})')
ax1.scatter([int(i) for i in test_indices], [0]*len(test_indices), c='red', s=15, marker='x', label=f'Test ({len(test_indices)})')
ax1.set_xlabel('Image index'); ax1.set_yticks([0,1]); ax1.set_yticklabels(['Test','Train'])
ax1.legend(); ax1.set_title('Train/Test 분포')

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(all_pos[train_mask,0], all_pos[train_mask,1], all_pos[train_mask,2], c='blue', s=8, alpha=0.5, label='Train')
ax2.scatter(all_pos[~train_mask,0], all_pos[~train_mask,1], all_pos[~train_mask,2], c='red', s=30, marker='x', label='Test')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.legend(); ax2.set_title('카메라 위치 (3D)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2) COLMAP 형식 변환 (train만 등록, --eval 미사용)
# ============================================================
if os.path.exists(SCENE_DIR):
    shutil.rmtree(SCENE_DIR)

images_dir = os.path.join(SCENE_DIR, "images")
sparse_dir = os.path.join(SCENE_DIR, "sparse")
test_images_dir = os.path.join(SCENE_DIR, "test_images")
os.makedirs(images_dir, exist_ok=True)
os.makedirs(sparse_dir, exist_ok=True)
os.makedirs(test_images_dir, exist_ok=True)

# cameras.txt
K0, _, _, W0, H0 = parse_calib(calib_dict[train_indices[0]])
fx, fy, cx, cy = K0[0, 0], K0[1, 1], K0[0, 2], K0[1, 2]

with open(os.path.join(sparse_dir, "cameras.txt"), 'w') as f:
    f.write("# Camera list with one line of data per camera:\n")
    f.write("# CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n")
    f.write(f"# Number of cameras: 1\n")
    f.write(f"1 PINHOLE {W0} {H0} {fx} {fy} {cx} {cy}\n")

# images.txt (train만)
camera_positions = []
with open(os.path.join(sparse_dir, "images.txt"), 'w') as f:
    f.write("# Image list with two lines of data per image:\n")
    f.write("# IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n")
    f.write("# POINTS2D[] as (X, Y, POINT3D_ID)\n")
    f.write(f"# Number of images: {len(train_indices)}\n")

    for img_id, idx in enumerate(train_indices, start=1):
        K, R, T, W, H = parse_calib(calib_dict[idx])
        qvec = rotmat2qvec(R)
        camera_positions.append(-R.T @ T)

        dst_name = f"rgb_{idx}.png"
        dst_path = os.path.join(images_dir, dst_name)
        if not os.path.exists(dst_path):
            os.symlink(os.path.abspath(rgb_dict[idx]), dst_path)

        f.write(f"{img_id} {qvec[0]} {qvec[1]} {qvec[2]} {qvec[3]} {T[0]} {T[1]} {T[2]} 1 {dst_name}\n")
        f.write("\n")

# test 이미지 복사
for idx in test_indices:
    dst_path = os.path.join(test_images_dir, f"rgb_{idx}.png")
    if not os.path.exists(dst_path):
        os.symlink(os.path.abspath(rgb_dict[idx]), dst_path)

camera_positions = np.array(camera_positions)
print(f"[OK] cameras.txt (PINHOLE {W0}x{H0}, fx={fx:.1f})")
print(f"[OK] images.txt (Train {len(train_indices)}장)")
print(f"[OK] test_images/ (Test {len(test_indices)}장)")

In [ ]:
# points3D.ply
from plyfile import PlyData, PlyElement

scene_center = camera_positions.mean(axis=0)
scene_radius = np.linalg.norm(camera_positions - scene_center, axis=1).mean()
obj_radius = scene_radius * 0.3

np.random.seed(42)
pts = []
while len(pts) < 1:
    batch = np.random.uniform(-1, 1, (NUM_INIT_POINTS * 2, 3))
    pts.append(batch[np.linalg.norm(batch, axis=1) <= 1.0])
pts = np.concatenate(pts)[:NUM_INIT_POINTS]
xyz = (pts * obj_radius + scene_center).astype(np.float32)
rgb = np.random.randint(0, 256, (NUM_INIT_POINTS, 3)).astype(np.uint8)

with open(os.path.join(sparse_dir, "points3D.txt"), 'w') as f:
    f.write("# 3D point list\n")

ply_path = os.path.join(sparse_dir, "points3D.ply")
dtype = [('x','f4'),('y','f4'),('z','f4'),('nx','f4'),('ny','f4'),('nz','f4'),('red','u1'),('green','u1'),('blue','u1')]
normals = np.zeros_like(xyz, dtype=np.float32)
elements = np.empty(NUM_INIT_POINTS, dtype=dtype)
elements[:] = list(map(tuple, np.concatenate([xyz, normals, rgb], axis=1)))
PlyData([PlyElement.describe(elements, 'vertex')]).write(ply_path)

print(f"[OK] points3D.ply ({NUM_INIT_POINTS:,} points)")
print(f"\n변환 완료! --eval 플래그 미사용 (직접 분할 완료)")

---
## 2. 학습

In [ ]:
#@title 학습 파라미터 {display-mode: "form"}

ITERATIONS = 30000  #@param {type:"integer"}
RESOLUTION = 2  #@param [1, 2, 4] {type:"raw"}
USE_APPEARANCE = True  #@param {type:"boolean"}

TGPC_WEIGHT = 0.1  #@param {type:"number"}
TGPC_NEIGHBORS = 12  #@param {type:"integer"}
MULTI_VIEW_NUM = 8  #@param {type:"integer"}
MULTI_VIEW_MAX_ANGLE = 30  #@param {type:"integer"}
MULTI_VIEW_MIN_DIS = 1.0  #@param {type:"number"}
MULTI_VIEW_MAX_DIS = 200.0  #@param {type:"number"}

LAMBDA_DSSIM = 0.2  #@param {type:"number"}
LAMBDA_DEPTH_NORMAL = 0.05  #@param {type:"number"}
LAMBDA_GEO = 0.03  #@param {type:"number"}
LAMBDA_NCC = 0.15  #@param {type:"number"}

SOURCE_PATH = SCENE_DIR
MODEL_PATH = f"{OUTPUT_DIR}/custom_scene"

print(f"Source: {SOURCE_PATH}")
print(f"Output: {MODEL_PATH}")

In [ ]:
os.chdir(WORK_DIR)

cmd = f"""python train.py \
    -s {SOURCE_PATH} \
    -m {MODEL_PATH} \
    -r {RESOLUTION} \
    --iterations {ITERATIONS} \
    --lambda_dssim {LAMBDA_DSSIM} \
    --lambda_depth_normal {LAMBDA_DEPTH_NORMAL} \
    --multi_view_geo_weight {LAMBDA_GEO} \
    --multi_view_ncc_weight {LAMBDA_NCC} \
    --tgpc_loss_weight {TGPC_WEIGHT} \
    --tgpc_num_neighbors {TGPC_NEIGHBORS} \
    --multi_view_num {MULTI_VIEW_NUM} \
    --multi_view_max_angle {MULTI_VIEW_MAX_ANGLE} \
    --multi_view_min_dis {MULTI_VIEW_MIN_DIS} \
    --multi_view_max_dis {MULTI_VIEW_MAX_DIS} \
    --test_iterations 7000 15000 30000 \
    --save_iterations 7000 30000"""

if USE_APPEARANCE:
    cmd += " --use_decoupled_appearance"

print(cmd)

In [ ]:
!{cmd}

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {MODEL_PATH}

---
## 3. 메쉬 추출

In [ ]:
os.chdir(WORK_DIR)
!python mesh_extract.py -s {SOURCE_PATH} -m {MODEL_PATH}

In [ ]:
import glob, trimesh
for mf in glob.glob(f"{MODEL_PATH}/**/fuse_*_post.ply", recursive=True) + \
          glob.glob(f"{MODEL_PATH}/**/recon.ply", recursive=True):
    mesh = trimesh.load(mf)
    print(f"{mf}")
    print(f"  Vertices: {len(mesh.vertices):,}, Faces: {len(mesh.faces):,}")

---
## 4. 렌더링 + 평가

In [ ]:
os.chdir(WORK_DIR)
# render.py는 mask가 없는 커스텀 데이터에서 크래시하므로 건너뜁니다.
# 대신 아래 셀에서 직접 test 뷰를 렌더링합니다.
# 원본 DTU/TNT 데이터에서는 아래 주석을 해제하세요:
# !python render.py -m {MODEL_PATH} --skip_test


In [ ]:
# Test 뷰 평가
import torchvision
import torch
import json
import numpy as np
from PIL import Image
import torchvision.transforms.functional as tf
from tqdm import tqdm

os.chdir(WORK_DIR)

from scene import GaussianModel
from gaussian_renderer import render as gs_render
from arguments import ModelParams, PipelineParams
from argparse import ArgumentParser
from scene.cameras import Camera as GSCamera
from utils.graphics_utils import focal2fov
from utils.loss_utils import ssim
from utils.image_utils import psnr
from lpipsPyTorch import lpips

parser = ArgumentParser()
lp = ModelParams(parser)
pp = PipelineParams(parser)
args = parser.parse_args(['-s', SOURCE_PATH, '-m', MODEL_PATH])
dataset = lp.extract(args)
pipe = pp.extract(args)


gaussians = GaussianModel(dataset.sh_degree)
from utils.system_utils import searchForMaxIteration
loaded_iter = searchForMaxIteration(os.path.join(MODEL_PATH, "point_cloud"))
gaussians.load_ply(os.path.join(MODEL_PATH, "point_cloud", f"iteration_{loaded_iter}", "point_cloud.ply"))
print(f"모델 로드: iteration {loaded_iter}, {gaussians.get_xyz.shape[0]:,} Gaussians")

bg_color = torch.tensor([0, 0, 0], dtype=torch.float32, device="cuda")
kernel_size = dataset.kernel_size

test_render_dir = os.path.join(MODEL_PATH, "test", f"ours_{loaded_iter}", "renders")
test_gt_dir = os.path.join(MODEL_PATH, "test", f"ours_{loaded_iter}", "gt")
os.makedirs(test_render_dir, exist_ok=True)
os.makedirs(test_gt_dir, exist_ok=True)

psnrs, ssims, lpipss = [], [], []

for ti, idx in enumerate(tqdm(test_indices, desc="Test rendering")):
    K, R, T, W, H = parse_calib(calib_dict[idx])
    scale = RESOLUTION
    W_s, H_s = W // scale, H // scale
    fx_s, fy_s = K[0,0] / scale, K[1,1] / scale
    FovX = focal2fov(fx_s, W_s)
    FovY = focal2fov(fy_s, H_s)

    gt_img = Image.open(rgb_dict[idx]).resize((W_s, H_s), Image.LANCZOS)
    gt_tensor = tf.to_tensor(gt_img).cuda()

    cam = GSCamera(
        colmap_id=ti, R=R.T, T=T,
        FoVx=FovX, FoVy=FovY,
        image=gt_tensor, gt_alpha_mask=None,
        image_name=f"test_{idx}", image_path=rgb_dict[idx], uid=ti,
        data_device="cuda"
    )

    with torch.no_grad():
        out = gs_render(cam, gaussians, pipe, bg_color, kernel_size=kernel_size)
        rendered = out["render"].clamp(0.0, 1.0)

    psnrs.append(psnr(rendered.unsqueeze(0), gt_tensor.unsqueeze(0)).item())
    ssims.append(ssim(rendered.unsqueeze(0), gt_tensor.unsqueeze(0)).item())
    lpipss.append(lpips(rendered.unsqueeze(0), gt_tensor.unsqueeze(0), net_type='vgg').item())

    torchvision.utils.save_image(rendered, os.path.join(test_render_dir, f"test_{idx}.png"))
    torchvision.utils.save_image(gt_tensor, os.path.join(test_gt_dir, f"test_{idx}.png"))

print(f"\n{'='*50}")
print(f"Test 평가 결과 ({len(test_indices)}장)")
print(f"{'='*50}")
print(f"  PSNR:  {np.mean(psnrs):.3f} dB")
print(f"  SSIM:  {np.mean(ssims):.4f}")
print(f"  LPIPS: {np.mean(lpipss):.4f}")

test_results = {
    "test_metrics": {
        "PSNR": float(np.mean(psnrs)),
        "SSIM": float(np.mean(ssims)),
        "LPIPS": float(np.mean(lpipss)),
        "num_test_views": len(test_indices),
        "num_train_views": len(train_indices),
        "iteration": loaded_iter
    }
}
with open(os.path.join(MODEL_PATH, "test_results.json"), 'w') as f:
    json.dump(test_results, f, indent=2)
print(f"저장: {MODEL_PATH}/test_results.json")

---
## 4.5 기하학적 정합도 평가

GT depth를 다운받아 렌더링 depth와 비교합니다.

| 지표 | 의미 | 좋은 값 |
|------|------|--------|
| AbsRel | 상대 오차 | 낮을수록 |
| RMSE | 절대 오차 | 낮을수록 |
| δ < 1.25 | 정확도 비율 | 높을수록 |
| Chamfer | 메쉬 표면 거리 | 낮을수록 |

In [ ]:
# GT Depth 다운로드 (기하학 평가용)
import gdown

DEPTH_ZIP = f"{DATA_DIR}/depth.zip"
DEPTH_SCALE = 0.01

if not os.path.exists(DEPTH_ZIP):
    gdown.download(
        "https://drive.google.com/uc?id=1KmXCzBYv_mkPmZnWka1ThCZSNHTyivKQ",
        DEPTH_ZIP, quiet=False
    )

!unzip -q -o {DEPTH_ZIP} -d {DATA_DIR}/raw_depth

def find_depth_root(base_dir, prefix):
    for root, dirs, files in os.walk(base_dir):
        if any(f.startswith(prefix) for f in files):
            return root
    return base_dir

DEPTH_DATA_DIR = find_depth_root(f"{DATA_DIR}/raw_depth", "depth_raw_")

from pathlib import Path
depth_dict = {Path(f).stem.split('_')[-1]: f
              for f in sorted(glob.glob(f"{DEPTH_DATA_DIR}/depth_raw_*.png"))}
print(f"Depth 파일: {len(depth_dict)}개")

In [ ]:
# 기하학적 정합도 평가: 렌더링 depth vs GT depth
import cv2

abs_rels, rmses = [], []
scale_factor = 1.0  # default if no valid views
delta_1, delta_2, delta_3 = [], [], []

for ti, idx in enumerate(tqdm(test_indices, desc="Depth evaluation")):
    K, R, T, W, H = parse_calib(calib_dict[idx])
    scale = RESOLUTION
    W_s, H_s = W // scale, H // scale
    fx_s, fy_s = K[0,0] / scale, K[1,1] / scale
    FovX = focal2fov(fx_s, W_s)
    FovY = focal2fov(fy_s, H_s)

    gt_img = Image.open(rgb_dict[idx]).resize((W_s, H_s), Image.LANCZOS)
    gt_tensor = tf.to_tensor(gt_img).cuda()

    cam = GSCamera(
        colmap_id=ti, R=R.T, T=T,
        FoVx=FovX, FoVy=FovY,
        image=gt_tensor, gt_alpha_mask=None,
        image_name=f"test_{idx}", image_path=rgb_dict[idx], uid=ti,
        data_device="cuda"
    )

    with torch.no_grad():
        out = gs_render(cam, gaussians, pipe, bg_color, kernel_size=kernel_size,
                       require_depth=True)
        rendered_depth = out["expected_depth"].squeeze(0).cpu().numpy()

    depth_src = depth_dict.get(idx)
    if depth_src is None:
        continue
    gt_depth = cv2.imread(depth_src, cv2.IMREAD_UNCHANGED).astype(np.float32) * DEPTH_SCALE
    gt_depth = cv2.resize(gt_depth, (W_s, H_s), interpolation=cv2.INTER_NEAREST)

    valid = gt_depth > 0
    if valid.sum() < 100:
        continue

    pred = rendered_depth[valid]
    gt = gt_depth[valid]

    # Median scaling alignment
    scale_factor = np.median(gt) / np.median(pred)
    pred = pred * scale_factor

    abs_rels.append(np.mean(np.abs(pred - gt) / gt))
    rmses.append(np.sqrt(np.mean((pred - gt) ** 2)))

    ratio = np.maximum(pred / gt, gt / pred)
    delta_1.append(np.mean(ratio < 1.25))
    delta_2.append(np.mean(ratio < 1.25 ** 2))
    delta_3.append(np.mean(ratio < 1.25 ** 3))

print(f"\n{'='*60}")
print(f"기하학적 정합도 ({len(abs_rels)}장 test 뷰)")
print(f"{'='*60}")
print(f"  Scale factor: {scale_factor:.4f}  (median alignment)")
print(f"  AbsRel:    {np.mean(abs_rels):.4f}")
print(f"  RMSE:      {np.mean(rmses):.4f}")
print(f"  δ < 1.25:  {np.mean(delta_1)*100:.2f}%")
print(f"  δ < 1.25²: {np.mean(delta_2)*100:.2f}%")
print(f"  δ < 1.25³: {np.mean(delta_3)*100:.2f}%")

geo_results = {
    "AbsRel": float(np.mean(abs_rels)),
    "RMSE": float(np.mean(rmses)),
    "delta_1.25": float(np.mean(delta_1)),
    "delta_1.25^2": float(np.mean(delta_2)),
    "delta_1.25^3": float(np.mean(delta_3)),
}
test_results["geometry_metrics"] = geo_results
with open(os.path.join(MODEL_PATH, "test_results.json"), 'w') as f:
    json.dump(test_results, f, indent=2)
print(f"저장: {MODEL_PATH}/test_results.json")

In [ ]:
# Depth 비교 시각화
n_vis = min(4, len(test_indices))
fig, axes = plt.subplots(n_vis, 3, figsize=(15, 5*n_vis))
if n_vis == 1: axes = axes[np.newaxis, :]

axes[0, 0].set_title('Rendered Depth', fontsize=14, fontweight='bold')
axes[0, 1].set_title('GT Depth', fontsize=14, fontweight='bold')
axes[0, 2].set_title('|Error|', fontsize=14, fontweight='bold')

for vi in range(n_vis):
    idx = test_indices[vi]
    K, R, T, W, H = parse_calib(calib_dict[idx])
    scale = RESOLUTION
    W_s, H_s = W // scale, H // scale
    fx_s, fy_s = K[0,0] / scale, K[1,1] / scale

    gt_img = Image.open(rgb_dict[idx]).resize((W_s, H_s), Image.LANCZOS)
    gt_tensor = tf.to_tensor(gt_img).cuda()

    cam = GSCamera(
        colmap_id=vi, R=R.T, T=T,
        FoVx=focal2fov(fx_s, W_s), FoVy=focal2fov(fy_s, H_s),
        image=gt_tensor, gt_alpha_mask=None,
        image_name=f"vis_{idx}", image_path=rgb_dict[idx], uid=vi,
        data_device="cuda"
    )

    with torch.no_grad():
        out = gs_render(cam, gaussians, pipe, bg_color, kernel_size=kernel_size,
                       require_depth=True)
        rd = out["expected_depth"].squeeze(0).cpu().numpy()

    gt_d = cv2.imread(depth_dict[idx], cv2.IMREAD_UNCHANGED).astype(np.float32) * DEPTH_SCALE
    gt_d = cv2.resize(gt_d, (W_s, H_s), interpolation=cv2.INTER_NEAREST)

    valid = gt_d > 0
    vmin, vmax = gt_d[valid].min(), gt_d[valid].max()
    sf = np.median(gt_d[valid]) / np.median(rd[valid]) if rd[valid].size > 0 and np.median(rd[valid]) > 0 else 1.0

    axes[vi, 0].imshow(rd * sf, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[vi, 0].set_ylabel(f'test_{idx}', fontsize=10)
    axes[vi, 0].axis('off')

    axes[vi, 1].imshow(gt_d, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[vi, 1].axis('off')

    rd_aligned = rd * sf
    error = np.abs(rd_aligned - gt_d) * valid
    axes[vi, 2].imshow(error, cmap='hot', vmin=0, vmax=np.percentile(error[valid], 95))
    axes[vi, 2].axis('off')

plt.suptitle(f'Depth 비교 (AbsRel: {np.mean(abs_rels):.4f})', fontsize=16)
plt.tight_layout()
plt.show()

### Pseudo GT 메쉬 vs 추출 메쉬 (Chamfer Distance)

In [ ]:
# Pseudo GT 메쉬 생성 → Chamfer Distance
from scipy.spatial import cKDTree
import open3d as o3d
import trimesh

pseudo_gt_points = []
for idx in tqdm(test_indices + train_indices, desc="Pseudo GT 생성"):
    depth_src = depth_dict.get(idx)
    if depth_src is None:
        continue
    K, R, T, W, H = parse_calib(calib_dict[idx])
    gt_d = cv2.imread(depth_src, cv2.IMREAD_UNCHANGED).astype(np.float32) * DEPTH_SCALE

    valid = gt_d > 0
    ys, xs = np.where(valid)
    if len(xs) > 1000:
        choice = np.random.choice(len(xs), 1000, replace=False)
        xs, ys = xs[choice], ys[choice]

    depths = gt_d[ys, xs]
    pts_cam = np.stack([
        (xs - K[0,2]) / K[0,0] * depths,
        (ys - K[1,2]) / K[1,1] * depths,
        depths
    ], axis=1)
    pts_world = (R.T @ (pts_cam.T - T[:, None])).T
    pseudo_gt_points.append(pts_world)

pseudo_gt = np.concatenate(pseudo_gt_points, axis=0).astype(np.float32)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pseudo_gt)
pcd = pcd.voxel_down_sample(voxel_size=0.5)
pseudo_gt = np.asarray(pcd.points).astype(np.float32)
print(f"Pseudo GT 포인트: {len(pseudo_gt):,}개")

mesh_files = sorted(glob.glob(f"{MODEL_PATH}/**/fuse_*_post.ply", recursive=True))
if not mesh_files:
    mesh_files = sorted(glob.glob(f"{MODEL_PATH}/**/fuse_*.ply", recursive=True))

if mesh_files:
    mesh = trimesh.load(mesh_files[0])
    pred_points = mesh.sample(min(200000, len(mesh.faces)))

    # 오브젝트 영역으로 크롭
    margin = 10.0
    bbox_min = pseudo_gt.min(axis=0) - margin
    bbox_max = pseudo_gt.max(axis=0) + margin
    mask = np.all((pred_points >= bbox_min) & (pred_points <= bbox_max), axis=1)
    pred_points = pred_points[mask]
    print(f"크롭 후 메쉬 포인트: {len(pred_points):,}개")

    tree_gt = cKDTree(pseudo_gt)
    d_pred_to_gt, _ = tree_gt.query(pred_points)
    tree_pred = cKDTree(pred_points)
    d_gt_to_pred, _ = tree_pred.query(pseudo_gt)

    chamfer = (d_pred_to_gt.mean() + d_gt_to_pred.mean()) / 2

    print(f"\n{'='*60}")
    print(f"Chamfer Distance (Pseudo GT 기준)")
    print(f"{'='*60}")
    print(f"  Chamfer Distance: {chamfer:.4f}")
    print(f"  Accuracy:         {d_pred_to_gt.mean():.4f}  (pred→gt)")
    print(f"  Completeness:     {d_gt_to_pred.mean():.4f}  (gt→pred)")

    test_results["chamfer_metrics"] = {
        "chamfer_distance": float(chamfer),
        "accuracy": float(d_pred_to_gt.mean()),
        "completeness": float(d_gt_to_pred.mean()),
    }
    with open(os.path.join(MODEL_PATH, "test_results.json"), 'w') as f:
        json.dump(test_results, f, indent=2)
    print(f"저장: {MODEL_PATH}/test_results.json")
else:
    print("메쉬 파일 없음. Section 3을 먼저 실행하세요.")

---
## 5. 시각화

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob, numpy as np

render_files = sorted(glob.glob(f"{test_render_dir}/*.png"))[:6]

if render_files:
    n = len(render_files)
    fig, axes = plt.subplots(n, 2, figsize=(12, 5*n))
    if n == 1: axes = axes[np.newaxis, :]

    axes[0, 0].set_title("Rendered", fontsize=14, fontweight='bold')
    axes[0, 1].set_title("Ground Truth", fontsize=14, fontweight='bold')

    for i, rf in enumerate(render_files):
        fname = os.path.basename(rf)
        axes[i, 0].imshow(Image.open(rf)); axes[i, 0].set_ylabel(fname); axes[i, 0].axis('off')
        gt = os.path.join(test_gt_dir, fname)
        if os.path.exists(gt): axes[i, 1].imshow(Image.open(gt))
        axes[i, 1].axis('off')

    plt.suptitle(f"Test Views (PSNR: {np.mean(psnrs):.2f} dB)", fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("렌더링 결과 없음")

In [ ]:
# Train 뷰 렌더링 확인
def show_train_results(model_path, max_images=4):
    iter_dirs = sorted(glob.glob(f"{model_path}/train/ours_*"))
    if not iter_dirs:
        print("Train 렌더링 없음")
        return
    d = iter_dirs[-1]
    renders = sorted(glob.glob(f"{d}/renders/*.png"))[:max_images]
    if not renders: return

    n = len(renders)
    fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
    if n == 1: axes = axes[np.newaxis, :]
    for j, t in enumerate(["Rendered", "Ground Truth", "Depth"]):
        axes[0, j].set_title(t, fontsize=14, fontweight='bold')
    for i, rf in enumerate(renders):
        axes[i, 0].imshow(Image.open(rf)); axes[i, 0].axis('off')
        gt = rf.replace('/renders/', '/gt/')
        if os.path.exists(gt): axes[i, 1].imshow(Image.open(gt))
        axes[i, 1].axis('off')
        depth = f"{d}/depths/{i:05d}.jpg"
        if os.path.exists(depth): axes[i, 2].imshow(Image.open(depth))
        axes[i, 2].axis('off')
    plt.tight_layout()
    plt.show()

show_train_results(MODEL_PATH)

---
## 6. 결과 저장

In [ ]:
# 결과를 zip으로 묶어서 다운로드
import json

# 실험 설정 기록
experiment_config = {
    "dataset": {
        "total_images": N_ALL,
        "num_train": len(train_indices),
        "num_test": len(test_indices),
        "resolution": RESOLUTION,
        "image_size": [W0, H0],
    },
    "training": {
        "iterations": ITERATIONS,
        "use_appearance": USE_APPEARANCE,
        "tgpc_weight": TGPC_WEIGHT,
        "tgpc_neighbors": TGPC_NEIGHBORS,
        "multi_view_num": MULTI_VIEW_NUM,
        "multi_view_max_angle": MULTI_VIEW_MAX_ANGLE,
        "multi_view_min_dis": MULTI_VIEW_MIN_DIS,
        "multi_view_max_dis": MULTI_VIEW_MAX_DIS,
        "lambda_dssim": LAMBDA_DSSIM,
        "lambda_depth_normal": LAMBDA_DEPTH_NORMAL,
        "lambda_geo": LAMBDA_GEO,
        "lambda_ncc": LAMBDA_NCC,
    },
    "test_metrics": test_results["test_metrics"],
}

config_path = os.path.join(MODEL_PATH, "experiment_config.json")
with open(config_path, 'w') as f:
    json.dump(experiment_config, f, indent=2)
print(f"실험 설정 저장: {config_path}")
print(json.dumps(experiment_config, indent=2))

In [ ]:
# 결과 zip 생성 (체크포인트 제외, 렌더링 + 메쉬 + 설정 + 메트릭만)
import zipfile

ZIP_RESULT = f"{OUTPUT_DIR}/triags_results.zip"

with zipfile.ZipFile(ZIP_RESULT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(MODEL_PATH):
        # 체크포인트(point_cloud, *.pth)는 용량이 크므로 제외
        if 'point_cloud' in root:
            continue
        for file in files:
            if file.endswith('.pth'):
                continue
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, OUTPUT_DIR)
            zf.write(full_path, arcname)

zip_size = os.path.getsize(ZIP_RESULT) / 1e6
print(f"zip 생성 완료: {ZIP_RESULT} ({zip_size:.1f} MB)")
print(f"\n포함된 내용:")
print(f"  - experiment_config.json (실험 설정 + 메트릭)")
print(f"  - test_results.json (PSNR/SSIM/LPIPS)")
print(f"  - test/ (test 뷰 렌더링 + GT)")
print(f"  - train/ (train 뷰 렌더링 + depth + normal)")
print(f"  - 메쉬 파일 (fuse_*.ply)")

In [ ]:
# Colab에서 zip 다운로드
try:
    from google.colab import files
    files.download(ZIP_RESULT)
except ImportError:
    print(f"로컬 환경입니다. 직접 복사하세요: {ZIP_RESULT}")

In [ ]:
# (선택) Google Drive에 저장
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/triags_results"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copy2(ZIP_RESULT, DRIVE_DIR)
    print(f"Drive 저장 완료: {DRIVE_DIR}/triags_results.zip")
else:
    print("Drive 저장 건너뜀")